# 실습 미션 1: Claude로 하던 업무를 파인튜닝으로 해결해 보기

가온반도체(가명) 설비팀에는 여섯 벤더의 설비에서 알람이 하루 5만 건씩 들어오고, 이 알람을 사내 Claude로 정규화하고 있습니다. 이 일을 작은 모델 E2B(파라미터 5.1B)를 파인튜닝해 처리하면 비용을 줄일 수 있을지 확인해 보려고 합니다.

알람은 벤더마다 포맷이 다릅니다.

**[벤더 알람 예시]**

```
A사   2026-10-16 22:05:05 [ALM][E-3677] CVD-12 CH2 LOADLOCK PUMP DOWN TIME EXCEEDED SEV=MINOR
B사   EQ=CVD-04|MOD=PM2|CODE=0x0072|LVL=2|MSG=MFC flow deviation
C사   {"tool": "PHOTO-05", "unit": "TM", "alarm": "STAGE_Y_SERVO_FOLLOWING_ERROR", "priority": "alarm", ...}
D사   [ETCH-02] 터보 펌프 회전 저하 (설비 중지) - 코드 S4956
E사   CMP-08 ALARM 5999 : TURBO PUMP SPEED LOW !! STOP
F사   #CMP-08 wafer not detected on end effector E:GS-536 [INFO]
```

지금은 사내 Claude에 벤더 알람마다 아래 **시스템 프롬프트**(19줄)를 함께 넣어 정규화하고 있습니다.

**[정규화 결과 예시]**

```
{"설비": "ETCH-02", "알람코드": "D-S4956", "부위": "진공", "심각도": "중지"}
```

그런데 알람 한 줄보다 시스템 프롬프트가 훨씬 길어서, 5만 건을 처리하면 **하루 약 227달러**가 듭니다.

<details><summary>시스템 프롬프트 펼쳐 보기 (system_prompt.txt)</summary>

```
[알람 표준 형식 v3]
벤더가 보낸 알람 메시지 한 건을 아래 네 칸의 JSON 한 줄로 바꾼다. JSON 만 답한다.
{"설비": "...", "알람코드": "...", "부위": "...", "심각도": "..."}

1. 설비: 메시지에 적힌 설비 이름 그대로 (예: ETCH-07)
2. 알람코드: 벤더 글자 + 하이픈 + 벤더의 알람 번호
3. 부위: 아래 여덟 가지 가운데 하나
   - 챔버: 챔버 압력 / RF: RF 전력, 매칭 / 진공: 배기, 펌프, 포어라인
   - 가스: 가스 유량, 가스 라인, 헬륨 / 이송: 로봇, 웨이퍼 감지, FOUP (로봇 흡착 해제도 이송)
   - 온도: 히터, 칠러, 벽 온도 / 약액: 약액, 탱크, 슬러리 / 스테이지: 스테이지
4. 심각도: 주의, 경고, 중지 가운데 하나

벤더별 표기 (알람코드 · 심각도)
- A사: [E-4021] → A-4021 · SEV=MINOR/MAJOR/CRITICAL → 주의/경고/중지
- B사: CODE=0x1F2A → B-1F2A (0x 는 뺀다) · LVL=1/2/3 → 주의/경고/중지
- C사: "alarm" 이름 → C-이름 그대로 (예: C-VAC_SLOW) · "priority" info/warn/alarm → 주의/경고/중지
- D사: 코드 W3307 → D-W3307 · (주의)/(경고)/(설비 중지) → 주의/경고/중지
- E사: ALARM 7710 → E-7710 · 끝 표시 「.」/「!」/「!! STOP」 → 주의/경고/중지
- F사: E:SV-031 → F-SV-031 · [INFO]/[WARN]/[CRIT] → 주의/경고/중지
```
</details>

이번 실습 미션에서는 지금까지 쌓인 Claude 정규화 결과로 학습셋을 만들어 E2B 모델을 파인튜닝하고, 새 알람 100건(평가셋)으로 정확도와 비용을 Claude와 비교합니다.

- 학습셋 재료 파일: `past_alarms.json`(지난 알람 2,000건), `claude_results.json`(그 Claude 정규화 결과)
- 평가셋 재료 파일: `evalset.json`(새 알람 100건과 사람이 확인한 정답)

| 단계 | 할 일 |
| --- | --- |
| 1 | 시스템 프롬프트만 넣었을 때 E2B의 정확도 확인 |
| 2 | Claude 정규화 결과로 학습셋 만들기 |
| 3 | 학습률과 에포크를 정해 LoRA로 파인튜닝하기 |
| 4 | 새 알람(평가셋)으로 파인튜닝 효과 확인하기 |
| 5 | Claude와 파인튜닝한 E2B의 하루 비용 비교하기 |
| 확장(선택) | 내 업무를 sLM 파인튜닝으로 해결해 보기 |

✏️ 표시가 있는 값은 기본값으로 먼저 실행하고, 주석의 선택지로 변경해 다시 실행합니다.

## 준비

라이브러리를 설치하고 E2B 모델과 재료를 불러옵니다. 설치 셀에서 실습 미션 1, 2, 3에 필요한 라이브러리를 한 번에 설치합니다.

In [ ]:
%pip install -q "transformers==5.18.0" "huggingface-hub<2" "peft==0.21.2" "accelerate==1.15.0" "bitsandbytes==0.50.2" matplotlib

In [ ]:
import os, warnings
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # 모델 내려받기 진행 막대와 안내 경고를 숨깁니다(결과와 상관없음)
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
import json, math, random, time
import torch
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'google/gemma-4-E2B-it'
random.seed(42)
torch.manual_seed(42)
from huggingface_hub import scan_cache_dir
from huggingface_hub.errors import CacheNotFound
try:                                          # 다른 미션에서 받은 모델 파일을 지워 디스크(40GB)를 비웁니다
    cache = scan_cache_dir()
    old = [rev.commit_hash for repo in cache.repos
           if repo.repo_id.startswith('google/gemma-4') and repo.repo_id != MODEL_NAME for rev in repo.revisions]
    if old:
        cache.delete_revisions(*old).execute()
except CacheNotFound:
    pass

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device_map='cuda')

system_prompt = open('mission_data/system_prompt.txt', encoding='utf-8').read()               # Claude에 넣고 있는 시스템 프롬프트
past_alarms = json.load(open('mission_data/past_alarms.json', encoding='utf-8'))              # 지난 알람 2,000건 (벤더, 메시지)
claude_results = json.load(open('mission_data/claude_results.json', encoding='utf-8'))        # 알람 id → Claude 정규화 결과
evalset = json.load(open('mission_data/evalset.json', encoding='utf-8'))                      # 평가셋 100건 (사람이 확인한 정답 포함)
claude_token_usage = json.load(open('mission_data/claude_token_usage.json', encoding='utf-8'))  # Claude 토큰 사용 기록

alarm = past_alarms[1]
print('지난 알람', len(past_alarms), '건 · 평가셋', len(evalset), '건')
print('알람        ', alarm)
print('Claude 결과 ', json.dumps(claude_results[alarm['id']], ensure_ascii=False))

In [ ]:
SHORT_PROMPT = '알람 메시지를 표준 형식 JSON 한 줄로 바꿔주세요.'   # 파인튜닝한 모델에 시스템 프롬프트 대신 넣는 짧은 지시
FIELDS = ['설비', '알람코드', '부위', '심각도']


def chat_text(head, alarm):
    """head(시스템 프롬프트 또는 짧은 지시) 뒤에 알람 한 건을 붙여 Gemma 대화 형식의 글로 만듭니다"""
    prompt = f"{head}\n벤더: {alarm['vendor']}\n메시지: {alarm['message']}\nJSON:"
    return tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True)


def parse_json(text):
    """생성된 글에서 JSON 부분을 파싱합니다. 실패하면 None (오답 처리)"""
    try:
        return json.loads(text[text.index('{'):text.rindex('}') + 1])
    except ValueError:
        return None


@torch.no_grad()
def normalize(model, rows, head, batch_size=10):
    """알람을 10건씩 묶어 정규화합니다. (결과 목록, 건당 처리 초)를 돌려줍니다"""
    model.eval()
    tokenizer.padding_side = 'left'          # 길이가 다른 프롬프트를 묶을 때 패딩을 왼쪽에 넣어야 프롬프트 끝에서 생성이 이어집니다
    outputs, start = [], time.perf_counter()
    for begin in range(0, len(rows), batch_size):
        texts = [chat_text(head, alarm) for alarm in rows[begin:begin + batch_size]]
        inputs = tokenizer(texts, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
        generated = model.generate(**inputs, max_new_tokens=60, do_sample=False)
        for sequence in generated:
            outputs.append(parse_json(tokenizer.decode(sequence[inputs['input_ids'].shape[1]:], skip_special_tokens=True)))
    tokenizer.padding_side = 'right'
    return outputs, (time.perf_counter() - start) / len(rows)


def score(outputs, rows):
    """네 필드가 모두 정답과 같은 비율(완전 일치, %)을 전체와 벤더별로 출력하고, 전체 비율을 돌려줍니다"""
    hits = [o is not None and all(o.get(f) == r['answer'][f] for f in FIELDS) for o, r in zip(outputs, rows)]
    by_vendor = defaultdict(list)
    for hit, row in zip(hits, rows):
        by_vendor[row['vendor']].append(hit)
    total = round(sum(hits) / len(rows) * 100)
    print(f'완전 일치 {total}%   |  ' + '  '.join(f'{v} {round(sum(h) / len(h) * 100)}%' for v, h in sorted(by_vendor.items())))
    return total


claude_score = score([claude_results[row['id']] for row in evalset], evalset)

## 1단계: E2B에 시스템 프롬프트를 넣어 정확도 확인

파인튜닝 없이 Claude와 같은 시스템 프롬프트만 넣어도 E2B로 정규화가 되는지 먼저 확인해 봅시다. 위 셀의 출력이 평가셋에서 Claude가 낸 정확도입니다. 같은 평가셋으로 E2B의 정확도를 확인합니다.

In [ ]:
base_outputs, _ = normalize(model, evalset, head=system_prompt)
base_score = score(base_outputs, evalset)

E2B가 정답과 다르게 답한 알람을 다섯 건 보여 줍니다. 시스템 프롬프트에 규칙이 모두 있어도 필드를 혼동하거나 형식이 깨진 출력이 많다면, 프롬프트만으로는 Claude를 대신하기 어렵다는 뜻입니다.

In [ ]:
for out, row in [(o, r) for o, r in zip(base_outputs, evalset) if o != r['answer']][:5]:
    print(f"[{row['vendor']}] {row['message'][:70]}\n   E2B   {json.dumps(out, ensure_ascii=False)}\n   정답 {json.dumps(row['answer'], ensure_ascii=False)}")

## 2단계: 학습셋 준비

지난 알람 300건과 그 Claude 정규화 결과로 학습셋을 만듭니다. 문제에는 시스템 프롬프트 대신 짧은 지시 한 줄을 넣습니다. 학습셋은 아래처럼 생겼습니다.

```
문제  알람 메시지를 표준 형식 JSON 한 줄로 바꿔주세요.
      벤더: B사
      메시지: EQ=CVD-04|MOD=PM2|CODE=0x0072|LVL=2|MSG=MFC flow deviation
정답  {"설비": "CVD-04", "알람코드": "B-0072", "부위": "가스", "심각도": "경고"}

문제  알람 메시지를 표준 형식 JSON 한 줄로 바꿔주세요.
      벤더: D사
      메시지: [ETCH-02] 터보 펌프 회전 저하 (설비 중지) - 코드 S4956
정답  {"설비": "ETCH-02", "알람코드": "D-S4956", "부위": "진공", "심각도": "중지"}

문제  알람 메시지를 표준 형식 JSON 한 줄로 바꿔주세요.
      벤더: E사
      메시지: CMP-08 ALARM 5999 : TURBO PUMP SPEED LOW !! STOP
정답  {"설비": "CMP-08", "알람코드": "E-5999", "부위": "진공", "심각도": "중지"}
```

아래 셀은 학습셋 300건을 만들고, 첫 건이 실제로 모델에 들어가는 모양(Gemma 대화 형식)으로 출력합니다.

In [ ]:
def make_example(alarm):
    """지난 알람 한 건 → 학습셋 한 건(문제: 짧은 지시 + 알람, 정답: Claude 정규화 결과)"""
    question = chat_text(SHORT_PROMPT, alarm)
    answer = json.dumps(claude_results[alarm['id']], ensure_ascii=False)
    return {'문제': question, '정답': answer}


train_alarms = random.Random(42).sample(past_alarms, 300)
examples = [make_example(alarm) for alarm in train_alarms]

print('── 문제 ──')
print(examples[0]['문제'])
print('── 정답 ──')
print(examples[0]['정답'])

## 3단계: 파인튜닝

예제 1-3과 같은 LoRA 학습입니다. 아래 `train_lora()`에서 파인튜닝의 핵심이 되는 코드는 다음과 같습니다.

| 코드 | 역할 |
| --- | --- |
| `LoraConfig(r=8, lora_alpha=16, ...)` | E2B 모델에 rank 8의 작은 행렬 두 개(LoRA 어댑터)를 더하도록 설정합니다 |
| `get_peft_model(model, config)` | E2B 모델에 어댑터를 추가합니다. 원래 가중치는 바뀌지 않고 어댑터 파라미터만 학습됩니다 |
| `AdamW(..., lr=learning_rate)` | 옵티마이저입니다. 손실이 줄어드는 쪽으로 어댑터 파라미터를 조금씩 업데이트하는 도구이고, 한 번에 얼마나 업데이트할지는 학습률로 정합니다 |
| `for epoch in range(epochs)` | 학습셋 300건 전체를 에포크 수만큼 반복합니다 |
| `tokenizer(texts, ...)` | 학습셋 4건(문제 + 정답)을 토큰으로 바꿔 한 배치로 묶습니다 |
| `labels[i, :len_question] = -100` | 문제 부분과 패딩은 손실 계산에서 빼고, 정답 JSON 부분만으로 손실을 계산하게 합니다. 모델은 문제를 따라 쓰지 않고 정답을 내도록 학습합니다 |
| `loss = tuned(**batch, labels=labels).loss` | 모델이 정답 JSON의 각 토큰에 매긴 확률로 손실을 계산합니다. 정답 토큰의 확률이 낮을수록 손실이 커집니다. 정답과 다르게 예측할수록 커집니다 |
| `loss.backward()` | 각 어댑터 파라미터를 어느 쪽으로 업데이트하면 손실이 줄어드는지(기울기)를 계산합니다 |
| `optimizer.step()` | 계산한 기울기를 써서 어댑터 파라미터를 실제로 업데이트합니다 |

In [ ]:
from peft import LoraConfig, get_peft_model


def train_lora(model, examples, learning_rate, epochs, batch_size=4):
    """학습셋으로 LoRA 어댑터를 학습합니다. (어댑터를 추가한 모델, 학습 초, 마지막 에포크 평균 손실)을 돌려줍니다"""
    config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type='CAUSAL_LM',
                        target_modules=r'.*language_model.*\.(q_proj|k_proj|v_proj|o_proj)')   # 예제 1-3과 같은 위치에 어댑터를 넣습니다
    tuned = get_peft_model(model, config)                      # 원래 가중치는 그대로 두고 어댑터 파라미터만 학습합니다
    optimizer = torch.optim.AdamW([p for p in tuned.parameters() if p.requires_grad], lr=learning_rate)
    tuned.train()
    start = time.perf_counter()
    for epoch in range(epochs):
        order = random.sample(examples, len(examples))         # 에포크마다 순서를 섞습니다
        losses = []
        for begin in range(0, len(order), batch_size):
            chunk = order[begin:begin + batch_size]
            texts = [ex['문제'] + ex['정답'] + tokenizer.eos_token for ex in chunk]   # eos_token: 답이 여기서 끝난다는 표시
            batch = tokenizer(texts, return_tensors='pt',
                              padding=True, truncation=True, max_length=256, add_special_tokens=False).to('cuda')
            labels = batch['input_ids'].clone()
            for i, ex in enumerate(chunk):
                len_question = len(tokenizer(ex['문제'], add_special_tokens=False)['input_ids'])
                labels[i, :len_question] = -100                # 문제 부분은 손실에서 뺍니다
            labels[batch['attention_mask'] == 0] = -100        # 패딩 자리도 손실에서 뺍니다
            loss = tuned(**batch, labels=labels).loss          # 손실 계산
            loss.backward()                                    # 기울기 계산
            optimizer.step()                                   # 어댑터 가중치 업데이트
            optimizer.zero_grad()
            losses.append(loss.item())
        epoch_loss = sum(losses) / len(losses)
        print(f'에포크 {epoch + 1} · 평균 손실 {epoch_loss:.4f}')
    return tuned, time.perf_counter() - start, epoch_loss


m1_records = []      # 학습하고 정확도를 확인할 때마다 한 줄씩 쌓이는 기록표

학습률과 학습 횟수(에포크)를 정해 E2B를 파인튜닝합니다. 먼저 예제 1-3과 같은 값(2e-4, 3)으로 학습해 봅시다.

In [ ]:
# ✏️ 직접 고칠 곳: 기본값(2e-4, 3)으로 먼저 학습하고 4단계에서 정확도를 확인한 뒤, Q1에 답하기 전에 아래 선택지로 값을 바꿔 한 번 이상 다시 학습합니다
LEARNING_RATE = 2e-4
# 5e-5, 2e-4, 1e-3 중 선택. 가중치를 한 번 업데이트할 때 변화 폭을 정합니다.
# 크면 손실이 빨리 줄지만 학습이 불안정해질 수 있고, 작으면 안정적이지만 같은 횟수 안에 덜 학습됩니다.

EPOCHS = 3
# 1, 3, 5 중 선택. 학습셋 300건 전체를 몇 번 반복해 학습할지 정합니다.
# 많을수록 학습셋에 잘 맞지만, 너무 많으면 학습셋에만 맞고 새 알람은 틀리는 과적합이 생길 수 있습니다.

if 'tuned' in globals():
    model = tuned.unload()       # 다시 학습할 때 이전 어댑터를 제거하고 원래 E2B로 되돌립니다
tuned, train_seconds, final_loss = train_lora(model, examples, LEARNING_RATE, EPOCHS)
print(f'학습 {train_seconds:.0f}초')

adapter_dir = 'adapters/alarm_' + MODEL_NAME.split('-')[-2]   # E2B로 학습하면 adapters/alarm_E2B, E4B면 adapters/alarm_E4B
tuned.save_pretrained(adapter_dir)     # 학습한 어댑터를 저장합니다. 실습 미션 3은 adapters/alarm_E2B에서 출발합니다
print('어댑터 저장', adapter_dir)

## 4단계: 파인튜닝한 E2B의 정확도 확인

파인튜닝한 E2B가 학습에 쓰지 않은 새 알람(평가셋)을 얼마나 정확하게 정규화하는지 확인해 봅시다. 목표는 완전 일치 95%입니다.

In [ ]:
tuned_outputs, tuned_sec = normalize(tuned, evalset, head=SHORT_PROMPT)
tuned_score = score(tuned_outputs, evalset)

m1_records.append({'학습률': LEARNING_RATE, '에포크': EPOCHS, '마지막 손실': round(final_loss, 4),
                '학습(초)': round(train_seconds), '완전 일치(%)': tuned_score})
print(f'\nClaude {claude_score}% · E2B 시스템 프롬프트 {base_score}%\n기록표')
for record in m1_records:
    print('   ', record)

3단계로 돌아가 학습률과 에포크를 변경해 다시 학습하면, 4단계를 실행할 때마다 기록표에 한 줄씩 추가됩니다.

**Q1.** 학습률과 에포크를 1회 이상 변경해 실행해 보고, 이 미션에서는 어느 값을 쓰는 것이 좋을지 기록표를 근거로 판단한 내용을 적어 주세요.

**답:** (이 칸을 두 번 클릭해 적습니다)

## 5단계: 하루 비용 비교

파인튜닝한 E2B로 바꾸면 비용을 얼마나 줄일 수 있는지 계산해 봅시다. Claude는 쓴 토큰만큼, E2B는 GPU를 켜 둔 시간만큼 비용이 듭니다. 실습 물량 범위에서는 E2B를 GPU 한 장으로 처리할 수 있으므로, E2B의 하루 비용은 GPU 한 장을 24시간 켜 둔 비용입니다. 같은 물량을 Claude로 처리할 때와 비교합니다.

In [ ]:
# ✏️ 직접 고칠 곳: 지금 물량(50_000)으로 먼저 계산하고, 1_000과 10_000으로 바꿔 다시 실행해 비용을 비교합니다
DAILY_VOLUME = 50_000
# 1_000, 10_000, 50_000 중 선택. 하루에 정규화할 알람 수입니다. 지금 물량은 50_000입니다.
# 여러 값으로 다시 실행해 물량에 따라 결과가 어떻게 달라지는지 확인합니다.

CLAUDE_INPUT_PRICE, CLAUDE_OUTPUT_PRICE = 4.0, 20.0      # Claude 100만 토큰당 가격(달러, 공개 단가)
GPU_PRICE_PER_HOUR = 0.89                                # RTX 4090 한 장 시간당 가격(달러)

claude_per_call = (claude_token_usage['input_tokens'] * CLAUDE_INPUT_PRICE
                   + claude_token_usage['output_tokens'] * CLAUDE_OUTPUT_PRICE) / 1e6 / claude_token_usage['calls']   # Claude 건당 비용
claude_daily = DAILY_VOLUME * claude_per_call
tuned_daily = 24 * GPU_PRICE_PER_HOUR                     # GPU 한 장을 24시간 켜 둔 비용
print(f'하루 {DAILY_VOLUME:,}건')
print(f'  Claude        {claude_daily:8,.2f}달러')
print(f'  E2B 파인튜닝   {tuned_daily:8,.2f}달러')
print(f'  차이          {claude_daily - tuned_daily:8,.2f}달러 ({(claude_daily - tuned_daily) / claude_daily * 100:+.0f}%)')

## 확장 미션(선택): 내 업무를 sLM 파인튜닝으로 해결해 보기

이번 미션처럼 지금 Claude로 처리하고 있지만, 파인튜닝한 작은 모델로 바꾸면 이득일 만한 업무를 하나 정합니다. 그 업무의 가상 재료를 Claude Code로 만들어 Claude와 파인튜닝한 E2B를 비교합니다.

1. 가상 재료를 Claude Code로 만듭니다. (예: 벤더별 알람)
2. 이 업무를 사내 Claude로 처리할 때 넣어야 하는 시스템 프롬프트를 적습니다.
3. 학습셋과 평가셋을 만듭니다.
4. 이 노트북의 함수로 E2B를 파인튜닝하고, 평가셋으로 Claude와 정확도와 비용을 비교합니다.

**회고:** 비교 결과 등 자유롭게 적어 주세요. (이 칸을 두 번 클릭해 적습니다)

# 실습 미션 2: GPU 메모리 한도 안에서 모델 크기와 정밀도 조합 비교하기

가온반도체(가명) R&D 팀은 부서 GPU 서버 한 장(RTX 4090, 24GB)에 계산과 추론 질문에 답하는 모델을 올리려고 합니다. E4B 모델은 정밀도를 낮추지 않고 그대로(bf16) 올릴 수 있지만, 여러 단계를 거쳐야 하는 문제에는 약할 수 있습니다. 더 큰 12B 모델은 정밀도를 낮춰야(양자화) GPU에 들어갑니다.

같은 GPU에서 어떤 조합이 문제를 가장 잘 푸는지, 2027학년도 수능 수학 문항으로 확인해 보려고 합니다.

**[후보 조합]**

아래 숫자는 모델마다 정밀도별로 올리는 데 필요한 메모리(GB)입니다. 실습 공간의 GPU는 24GB이고, 그 80%인 19.2GB 이하인 조합만 올릴 수 있습니다.

```
모델    파라미터    bf16       int8       fp4
E2B     5.1B      9.5GB      4.7GB      2.4GB
E4B     8.0B     14.9GB      7.5GB      3.7GB
12B    12.0B     22.4GB     11.2GB      5.6GB
```

**[문항 예시]**

```
1번 (객관식, 2점)  ∛32 × 4^(-1/3) 의 값은?   ① 1  ② 2  ③ 3  ④ 4  ⑤ 5        정답 ②
17번 (단답형, 3점) 다항함수 f(x) 에 대하여 f'(x) = 6x² + 1, f(1) = 5 가 성립할 때, f(2) 의 값을 구하시오.   정답 20
```

이번 실습 미션에서는 후보 조합을 올려 15문항을 풀게 하고, 정답률과 메모리, 속도를 비교합니다.

- 문항 재료 파일: `mission_data/suneung2027_math.json` (그림이 없는 2·3점 문항 15개와 정답)
- 출처: 한국교육과정평가원, 2027학년도 대학수학능력시험 수학 영역(홀수형)

| 단계 | 단계별 학습 내용 |
| --- | --- |
| 1 | GPU 메모리로 들어가는 조합 계산 |
| 2 | 조합마다 수능 문항을 풀게 하고 정답률, 메모리, 속도 측정 |
| 참고 | 같은 메모리 한도에서 조합 비교 |
| 확장(선택) | 미션 1의 알람 정규화로 같은 비교 해 보기 |

✏️ 표시가 있는 값은 기본값으로 먼저 실행하고, 주석의 선택지로 변경해 다시 실행합니다.

## 준비

미션 1에서 올린 모델을 GPU에서 내리고, 문항을 불러옵니다.

In [ ]:
# 앞 미션에서 올린 모델을 GPU에서 내립니다. 이어서 올릴 모델의 메모리가 모자라지 않게 합니다
import gc, torch
for name in ['model', 'tuned', 'base']:
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print('지금 GPU 에 올라간 것', round(torch.cuda.memory_allocated() / 1024**3, 1), 'GB')

In [ ]:
import os, warnings
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # 모델 내려받기 진행 막대와 안내 경고를 숨깁니다(결과와 상관없음)
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
import gc, json, re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_IDS = {'E2B': 'google/gemma-4-E2B-it', 'E4B': 'google/gemma-4-E4B-it',
             '12B': 'google/gemma-4-12B-it'}
PARAMS_BILLION = {'E2B': 5.1, 'E4B': 8.0, '12B': 12.0}

problems = json.load(open('mission_data/suneung2027_math.json', encoding='utf-8'))   # 문항 15개 (출처: 한국교육과정평가원)
print('문항', len(problems), '개')
print(problems[0]['question'], problems[0]['choices'], '정답', problems[0]['answer'])

In [ ]:
def make_prompt(problem):
    """문항 하나를 프롬프트로 만듭니다. 풀이 뒤 마지막 줄에 정답만 적게 해 자동으로 채점합니다"""
    text = problem['question']
    if problem['choices']:
        text += '\n' + '\n'.join(f'{i + 1}) {choice}' for i, choice in enumerate(problem['choices']))
        rule = '풀이를 적은 뒤 마지막 줄에 「정답: 번호」(1~5 중 하나)만 적으세요.'
    else:
        rule = '풀이를 적은 뒤 마지막 줄에 「정답: 정수」만 적으세요.'
    return f'{text}\n\n{rule}'


def extract_answer(text):
    """생성된 풀이에서 마지막 「정답: 숫자」를 추출합니다. 없으면 None (오답 처리)"""
    found = re.findall(r'정답\s*[:：]\s*(-?\d+)', text)
    return found[-1] if found else None


@torch.no_grad()
def solve(model, tokenizer, problems, max_new_tokens=768, batch_size=5):
    """문항을 5개씩 묶어 풀게 합니다. (풀이 목록, 문항당 초)를 돌려줍니다"""
    model.eval()
    tokenizer.padding_side = 'left'
    solutions, start = [], time.perf_counter()
    for begin in range(0, len(problems), batch_size):
        texts = [tokenizer.apply_chat_template([{'role': 'user', 'content': make_prompt(p)}], tokenize=False,
                                               add_generation_prompt=True) for p in problems[begin:begin + batch_size]]
        inputs = tokenizer(texts, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        for sequence in generated:
            solutions.append(tokenizer.decode(sequence[inputs['input_ids'].shape[1]:], skip_special_tokens=True))
    return solutions, (time.perf_counter() - start) / len(problems)

## 1단계: GPU 메모리로 들어가는 조합 계산

모델을 올리기 전에, 파라미터 수와 정밀도만으로 어느 조합이 GPU에 들어가는지 계산해 봅시다. 예제 2-1처럼 GPU 메모리의 80%를 한도로 두고, 나머지 20%는 답을 생성하는 동안 필요한 여유분으로 남깁니다.

In [ ]:
# ✏️ 직접 고칠 곳: 실습 공간 GPU(24)로 먼저 실행하고, 16이나 48로 바꿔 들어가는 조합이 어떻게 달라지는지 확인합니다
GPU_MEMORY_GB = 24
# 16, 24, 48 중 선택. 모델을 올릴 GPU의 메모리 크기(GB)입니다.
# 24는 지금 실습 공간의 RTX 4090입니다. 다른 값을 넣으면 GPU가 바뀔 때 고를 수 있는 조합이 어떻게 달라지는지 볼 수 있습니다.

limit_gb = GPU_MEMORY_GB * 0.8
bytes_per_value = {'bf16': 2, 'int8': 1, 'fp4': 0.5}       # 정밀도별 값 하나의 바이트 수

print(f'GPU {GPU_MEMORY_GB}GB · 한도 {limit_gb:.1f}GB\n')
print('모델   ' + '   '.join(f'{p:>7s}' for p in bytes_per_value))
for model_size, params in PARAMS_BILLION.items():
    cells_text = []
    for precision, size in bytes_per_value.items():
        size_gb = params * 1e9 * size / 1024**3                # 파라미터 수 × 값 하나의 바이트 수
        cells_text.append(f'{size_gb:5.1f}GB' + (' ' if size_gb <= limit_gb else '×'))
    print(f'{model_size:5s}  ' + '  '.join(cells_text))
print('\n× 는 한도를 넘어 올릴 수 없는 조합입니다')

## 2단계: 조합마다 수능 문항 풀게 하기

1단계에서 실습 공간(24GB)에 들어가는 조합을 골라 올리고, 15문항을 풀게 해 정답률과 메모리, 문항당 시간을 측정해 봅시다. 측정 결과는 기록표에 한 줄씩 쌓입니다. E4B를 bf16으로 먼저 측정하고, 더 큰 12B를 fp4로 줄여 올려 비교합니다.

모델을 정밀도에 맞춰 올리는 핵심 코드는 다음과 같습니다.

| 코드 | 역할 |
| --- | --- |
| `BitsAndBytesConfig(load_in_8bit=True)` | 가중치를 값 하나당 1바이트(int8)로 줄여 올리도록 설정합니다 |
| `BitsAndBytesConfig(load_in_4bit=True, ...)` | 가중치를 값 하나당 0.5바이트(4비트 부동소수점, fp4)로 줄여 올리도록 설정합니다. 계산할 때는 bf16으로 되돌려 계산합니다 |
| `from_pretrained(..., quantization_config=quant)` | 설정한 정밀도로 모델을 GPU에 올립니다. bf16이면 설정 없이 올립니다 |
| `torch.cuda.memory_allocated()` | 지금 GPU에 올라간 크기를 읽습니다 |
| `model.generate(..., max_new_tokens=768)` | 풀이를 최대 768토큰까지 생성합니다. 풀이가 이보다 길면 정답 줄이 잘려 오답이 됩니다 |

In [ ]:
from huggingface_hub import scan_cache_dir
from huggingface_hub.errors import CacheNotFound

BYTES_PER_VALUE = {'bf16': 2, 'int8': 1, 'fp4': 0.5}


def check_fits(model_size, precision):
    """조합이 이 GPU 메모리의 80% 안에 드는지 먼저 계산합니다. 들어가지 않으면 오류 대신 안내를 출력하고 False를 돌려줍니다"""
    if model_size not in MODEL_IDS or precision not in BYTES_PER_VALUE:
        print(f"'{model_size}', '{precision}'은 선택지에 없습니다. 모델은 {list(MODEL_IDS)}, 정밀도는 {list(BYTES_PER_VALUE)} 중에서 고릅니다.")
        return False
    need_gb = PARAMS_BILLION[model_size] * 1e9 * BYTES_PER_VALUE[precision] / 1024**3
    limit_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 * 0.8
    if need_gb > limit_gb:
        print(f'{model_size} {precision}은 약 {need_gb:.1f}GB가 필요해 이 GPU 한도({limit_gb:.1f}GB)에 들어가지 않습니다. 1단계 표에서 ×가 아닌 조합을 고릅니다.')
        return False
    return True


def free_disk(keep_model_size):
    """지금 올릴 모델을 뺀 나머지 후보 모델의 내려받은 파일을 지웁니다. 모델을 여러 개 받으면 디스크(40GB)가 가득 차기 때문입니다"""
    keep = MODEL_IDS[keep_model_size]
    try:
        cache = scan_cache_dir()
    except CacheNotFound:                                       # 아직 내려받은 모델이 없으면 지울 것도 없습니다
        return
    old = [rev.commit_hash for repo in cache.repos
           if repo.repo_id in MODEL_IDS.values() and repo.repo_id != keep for rev in repo.revisions]
    if old:
        cache.delete_revisions(*old).execute()


def load(model_size, precision):
    """모델 크기와 정밀도에 맞춰 모델을 올립니다. (모델, 토크나이저, 올린 뒤 메모리 GB)를 돌려줍니다"""
    free_disk(model_size)
    if precision == 'int8':
        quant = BitsAndBytesConfig(load_in_8bit=True)
    elif precision == 'fp4':
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='fp4', bnb_4bit_compute_dtype=torch.bfloat16)
    else:
        quant = None                                            # bf16은 양자화 없이 올립니다
    tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS[model_size])
    model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[model_size], dtype=torch.bfloat16,
                                                 device_map='cuda', quantization_config=quant)
    return model, tokenizer, torch.cuda.memory_allocated() / 1024**3


def unload():
    """앞에서 올린 모델을 GPU 에서 내립니다. 측정마다 빈 GPU 에서 시작해야 메모리 값이 섞이지 않습니다"""
    globals().pop('model', None)
    gc.collect()
    torch.cuda.empty_cache()


m2_records = []      # 측정할 때마다 한 줄씩 쌓이는 기록표

In [ ]:
# ✏️ 직접 고칠 곳: E4B bf16으로 먼저 측정하고, 12B fp4로 바꿔 다시 측정합니다
MODEL_SIZE = 'E4B'
# 'E2B', 'E4B', '12B' 중 선택. 올릴 모델입니다. 클수록 파라미터가 많아 여러 단계를 거치는 문제에 강할 수 있지만 메모리를 더 씁니다.

PRECISION = 'bf16'
# 'bf16', 'int8', 'fp4' 중 선택. 가중치 값 하나를 몇 바이트로 담을지 정합니다.
# 낮출수록 메모리가 줄지만, 가중치 하나가 가질 수 있는 값의 개수가 줄어(8비트 256개, 4비트 16개) 원래 값과의 차이(양자화 오차)가 커지고 정답률이 떨어질 수 있습니다.
# 비교할 조합: E4B bf16(14.9GB)과 12B fp4(5.6GB)

assert check_fits(MODEL_SIZE, PRECISION), '위 안내를 보고 MODEL_SIZE와 PRECISION을 다시 고릅니다'
unload()
model, tokenizer, memory_gb = load(MODEL_SIZE, PRECISION)
solutions, seconds = solve(model, tokenizer, problems)
correct = [extract_answer(s) == p['answer'] for s, p in zip(solutions, problems)]

m2_records.append({'모델': MODEL_SIZE, '정밀도': PRECISION, '메모리(GB)': round(memory_gb, 1),
                '정답': f'{sum(correct)}/{len(problems)}', '문항당(초)': round(seconds, 1)})
print('문항별 ·', ' '.join(f"{p['no']}{'O' if c else 'X'}" for p, c in zip(problems, correct)))
print('\n기록표')
for record in m2_records:
    print('   ', record)

틀린 문항의 풀이를 하나 출력해 봅니다. 계산 실수인지, 정답 줄을 적기 전에 풀이가 잘렸는지 확인할 수 있습니다.

In [ ]:
wrong = [i for i, c in enumerate(correct) if not c]
if wrong:
    i = wrong[0]
    print(f"{problems[i]['no']}번 · 정답 {problems[i]['answer']}\n")
    print(solutions[i][-800:])

모델과 정밀도를 변경해 위 셀을 다시 실행하면 기록표에 한 줄씩 추가됩니다.

**Q1.** 기록표에서 E4B bf16과 12B fp4를 비교해 주세요. 작은 모델을 높은 정밀도로 쓰는 것과 큰 모델을 낮은 정밀도로 쓰는 것 가운데 이번 결과에서는 어느 쪽이 나았는지 판단한 내용을 적어 주세요.

**답:** (이 칸을 두 번 클릭해 적습니다)

## [참고] 같은 메모리 한도에서 비교

메모리 한도를 정하면, 그 안에 들어가는 모든 조합을 계산으로 찾고 2단계에서 측정한 결과를 함께 보여 줍니다. 같은 GPU를 쓸 때 어떤 조합을 올리는 것이 나은지 확인해 봅시다. 아직 측정하지 않은 조합은 「측정 안 함」으로 나오니, 필요하면 2단계로 돌아가 측정합니다.

In [ ]:
# [참고] 한도를 바꿔 실행하면 그 안에 드는 조합과 2단계 측정 결과를 함께 볼 수 있습니다
MEMORY_LIMIT_GB = 16
# 8, 12, 16 중 선택. 모델에 쓸 수 있는 메모리 한도(GB)입니다.
# 값을 변경해 가며 한도에 따라 고를 수 있는 조합이 어떻게 달라지는지 확인합니다.

measured = {(r['모델'], r['정밀도']): r for r in m2_records}
print(f'메모리 {MEMORY_LIMIT_GB}GB 이하에 들어가는 조합\n')
for model_size, params in PARAMS_BILLION.items():
    for precision, size in bytes_per_value.items():
        size_gb = params * 1e9 * size / 1024**3
        if size_gb > MEMORY_LIMIT_GB:
            continue
        record = measured.get((model_size, precision))
        result = f"정답 {record['정답']} · 문항당 {record['문항당(초)']}초" if record else '측정 안 함'
        print(f'   {model_size:4s} {precision:5s}  계산 {size_gb:5.1f}GB  →  {result}')

## 확장 미션(선택): 미션 1의 알람 정규화로 같은 비교 해 보기

수능 수학처럼 여러 단계를 거치는 문제와 달리, 미션 1의 알람 정규화는 정해진 형식으로 바꾸는 과제입니다. 과제가 바뀌어도 「큰 모델 + 낮은 정밀도」가 나은지, 미션 1을 모델 크기와 정밀도만 변경해 다시 해 보고 비교합니다.

1. 미션 1 준비 셀의 `MODEL_NAME`을 `google/gemma-4-E4B-it`로 바꿔 미션 1을 다시 실행합니다. 어댑터는 모델 이름에 따라 `adapters/alarm_E4B`로 따로 저장되므로, 미션 3이 쓰는 `adapters/alarm_E2B`는 그대로 남습니다.
2. `from peft import PeftModel`을 먼저 실행합니다. 이 노트북의 `load()`로 원본 모델을 정밀도별로 올리고 `PeftModel.from_pretrained(model, 'adapters/alarm_E4B')`로 어댑터를 붙인 뒤, 미션 1의 평가셋으로 완전 일치를 측정합니다.
3. 조합별로 미션 2 수학 풀이 순위와 미션 1 정규화 순위를 비교해 봅니다.

**회고:** 비교 결과 등 자유롭게 적어 주세요. (이 칸을 두 번 클릭해 적습니다)

# 실습 미션 3: 새 벤더 알람을 추가 학습하고 망각 점검하기

가온반도체(가명) 설비팀은 미션 1처럼 A\~F사 알람을 파인튜닝한 E2B 모델로 정규화하고 있습니다. 그런데 이번 달에 G사 설비가 새로 들어왔고, G사 알람은 A~F사와 또 다른 포맷으로 구성되어 있습니다.

**[G사 알람 예시]**

```
<evt eq="ETCH-02" id="425" sev="3">chiller return temp high</evt>   →  {"설비": "ETCH-02", "알람코드": "G-425", "부위": "온도", "심각도": "경고"}
<evt eq="ETCH-07" id="986" sev="1">MFC flow deviation</evt>          →  {"설비": "ETCH-07", "알람코드": "G-986", "부위": "가스", "심각도": "주의"}
```

Claude로 정규화한다면 시스템 프롬프트에 G사 표기 규칙 한 줄만 추가하면 해결됩니다.

```
- G사: id="882" → G-882 · sev="1"/"3"/"5" → 주의/경고/중지
```

하지만 파인튜닝한 E2B는 G사 학습셋을 추가로 학습해야 합니다. 이번 실습 미션에서는 이처럼 새로운 학습셋을 추가해야 할 때 G사 학습셋을 몇 건이나 추가해야 하는지 학습 곡선으로 확인하고, G사를 학습하는 동안 기존 A\~F사 정확도가 떨어지지 않는지(망각) 함께 점검합니다.

- 지금 쓰는 모델: `adapters/alarm_E2B` (실습 미션 1에서 A\~F사 학습셋으로 파인튜닝해 저장한 어댑터, 없으면 강사가 미리 학습해 둔 `adapters/instructor_alarm_E2B`)
- G사 학습셋 재료: `mission_data/g_alarms.json` (G사 알람 200건과, G사 규칙을 추가한 시스템 프롬프트로 받은 Claude 정규화 결과)
- 평가셋: `mission_data/g_evalset.json` (G사 새 알람 60건), `mission_data/evalset.json` (미션 1의 A\~F사 새 알람 100건)
- 기존 학습셋 재료: `mission_data/past_alarms.json`, `mission_data/claude_results.json` (미션 1과 같음)

| 단계 | 단계별 학습 내용 |
| --- | --- |
| 1 | 지금 모델의 G사, A\~F사 정확도 확인 |
| 2 | G사 학습셋을 조금씩 늘려 학습하고 학습 곡선 확인 |
| 3 | 기존 학습셋을 섞어 학습하고 망각 비교 |
| 확장(선택) | 내 업무에 새 유형이 추가된 경우로 해 보기 |

✏️ 표시가 있는 값은 기본값으로 먼저 실행하고, 주석의 선택지로 변경해 다시 실행합니다.

## 준비

미션 2에서 올린 모델을 GPU에서 내리고, E2B 모델과 재료를 불러옵니다.

In [ ]:
# 앞 미션에서 올린 모델을 GPU에서 내립니다. 이어서 올릴 모델의 메모리가 모자라지 않게 합니다
import gc, torch
for name in ['model', 'tuned', 'base']:
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print('지금 GPU 에 올라간 것', round(torch.cuda.memory_allocated() / 1024**3, 1), 'GB')

In [ ]:
import os, warnings
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # 모델 내려받기 진행 막대와 안내 경고를 숨깁니다(결과와 상관없음)
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
import json, os, random
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = 'google/gemma-4-E2B-it'
SHORT_PROMPT = '알람 메시지를 표준 형식 JSON 한 줄로 바꿔주세요.'     # 미션 1에서 학습할 때 넣은 짧은 지시
FIELDS = ['설비', '알람코드', '부위', '심각도']
random.seed(42)
torch.manual_seed(42)

from huggingface_hub import scan_cache_dir
from huggingface_hub.errors import CacheNotFound
try:                                          # 다른 미션에서 받은 모델 파일을 지워 디스크(40GB)를 비웁니다
    cache = scan_cache_dir()
    old = [rev.commit_hash for repo in cache.repos
           if repo.repo_id.startswith('google/gemma-4') and repo.repo_id != MODEL_NAME for rev in repo.revisions]
    if old:
        cache.delete_revisions(*old).execute()
except CacheNotFound:
    pass

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device_map='cuda')

g_alarms = json.load(open('mission_data/g_alarms.json', encoding='utf-8'))              # G사 알람 200건 + Claude 정규화 결과
g_evalset = json.load(open('mission_data/g_evalset.json', encoding='utf-8'))            # G사 새 알람 60건 (정답 포함)
old_evalset = json.load(open('mission_data/evalset.json', encoding='utf-8'))            # A~F사 새 알람 100건 (정답 포함)
past_alarms = json.load(open('mission_data/past_alarms.json', encoding='utf-8'))        # A~F사 지난 알람 2,000건
claude_results = json.load(open('mission_data/claude_results.json', encoding='utf-8'))  # 알람 id → Claude 정규화 결과
print('G사 학습셋 재료', len(g_alarms), '건 · G사 평가셋', len(g_evalset), '건 · A~F사 평가셋', len(old_evalset), '건')

In [ ]:
def chat_text(head, alarm):
    prompt = f"{head}\n벤더: {alarm['vendor']}\n메시지: {alarm['message']}\nJSON:"
    return tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False, add_generation_prompt=True)


def parse_json(text):
    try:
        return json.loads(text[text.index('{'):text.rindex('}') + 1])
    except ValueError:
        return None


@torch.no_grad()
def accuracy(model, rows, batch_size=10):
    """rows 를 정규화해 네 필드가 모두 정답과 같은 비율(완전 일치, %)을 돌려줍니다"""
    model.eval()
    tokenizer.padding_side = 'left'
    hits = []
    for begin in range(0, len(rows), batch_size):
        chunk = rows[begin:begin + batch_size]
        inputs = tokenizer([chat_text(SHORT_PROMPT, a) for a in chunk], return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
        generated = model.generate(**inputs, max_new_tokens=60, do_sample=False)
        for sequence, row in zip(generated, chunk):
            out = parse_json(tokenizer.decode(sequence[inputs['input_ids'].shape[1]:], skip_special_tokens=True))
            hits.append(out is not None and all(out.get(f) == row['answer'][f] for f in FIELDS))
    return round(sum(hits) / len(rows) * 100)


def to_example(alarm, answer):
    """학습셋 한 건: 문제(짧은 지시 + 알람)와 정답(정규화 결과)"""
    return {'문제': chat_text(SHORT_PROMPT, alarm), '정답': json.dumps(answer, ensure_ascii=False)}


def train(model, examples, learning_rate=2e-4, epochs=3, batch_size=4):
    """미션 1 과 같은 방식으로 학습합니다. 정답 부분만 손실에 넣습니다"""
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=learning_rate)
    model.train()
    tokenizer.padding_side = 'right'
    for epoch in range(epochs):
        order = random.sample(examples, len(examples))
        for begin in range(0, len(order), batch_size):
            chunk = order[begin:begin + batch_size]
            batch = tokenizer([ex['문제'] + ex['정답'] + tokenizer.eos_token for ex in chunk], return_tensors='pt',
                              padding=True, truncation=True, max_length=256, add_special_tokens=False).to('cuda')
            labels = batch['input_ids'].clone()
            for i, ex in enumerate(chunk):
                labels[i, :len(tokenizer(ex['문제'], add_special_tokens=False)['input_ids'])] = -100   # 문제 부분은 손실에서 뺍니다
            labels[batch['attention_mask'] == 0] = -100                                                # 패딩 자리도 뺍니다
            loss = model(**batch, labels=labels).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
    return model

## 1단계: 지금 모델의 정확도 확인

A\~F사로만 학습한 지금 모델이 처음 보는 G사 알람을 얼마나 정확하게 정규화하는지, 기존 A\~F사 정확도와 함께 확인해 봅시다.

In [ ]:
ADAPTER_DIR = 'adapters/alarm_E2B'                       # 실습 미션 1에서 저장한 어댑터
if not os.path.exists(ADAPTER_DIR):
    ADAPTER_DIR = 'adapters/instructor_alarm_E2B'         # 미션 1을 끝내지 못했으면 강사가 미리 학습해 둔 어댑터를 씁니다
print('출발 어댑터', ADAPTER_DIR)

model = PeftModel.from_pretrained(base, ADAPTER_DIR)     # 원본 E2B에 미션 1 어댑터를 붙입니다
start_g = accuracy(model, g_evalset)
start_old = accuracy(model, old_evalset)
print(f'G사 {start_g}% · A~F사 {start_old}%')

## 2단계: G사 학습셋을 늘려 가며 학습하기

G사 학습셋을 몇 건 추가해야 G사 정확도가 충분히 오르는지 확인해 봅시다. 건수를 변경해 실행할 때마다 지금 모델(미션 1 어댑터)에서 새로 시작해 G사 학습셋만 추가로 학습하고, G사와 A\~F사 정확도를 기록표에 추가합니다. 두 번 이상 실행하면 학습 곡선이 그려집니다.

이미 학습한 모델을 이어서 학습하는 핵심 코드는 다음과 같습니다.

| 코드 | 역할 |
| --- | --- |
| `PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=True)` | 미션 1 어댑터를 불러오면서 학습할 수 있는 상태로 둡니다. 새로 만드는 대신 이미 학습한 어댑터에서 이어서 학습합니다 |
| `model.unload()` | 붙어 있는 어댑터를 떼어 원본 E2B로 되돌립니다. 다시 실행할 때 같은 출발점에서 시작하려고 씁니다 |
| `train(model, examples)` | 미션 1과 같은 방식으로 학습합니다(학습률 2e-4, 3에포크, 정답 부분만 손실) |

In [ ]:
m3_records = []      # 학습할 때마다 한 줄씩 쌓이는 기록표


def update_and_check(g_count, old_count):
    """미션 1 어댑터를 다시 불러와 G사 g_count 건과 기존 A~F사 old_count 건으로 추가 학습하고, 두 정확도를 기록합니다"""
    global model, base
    base = model.unload()                                                      # 앞 실행의 어댑터를 떼고 원본으로
    model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=True)
    examples = [to_example(a, a['claude_result']) for a in g_alarms[:g_count]]
    examples += [to_example(a, claude_results[a['id']]) for a in random.Random(7).sample(past_alarms, old_count)]
    train(model, examples)
    record = {'G사 건수': g_count, '기존 건수': old_count,
              'G사(%)': accuracy(model, g_evalset), 'A~F사(%)': accuracy(model, old_evalset)}
    m3_records.append(record)
    print(f"학습 전 · G사 {start_g}% · A~F사 {start_old}%")
    for r in m3_records:
        print('   ', r)

In [ ]:
# ✏️ 직접 고칠 곳: 5건으로 먼저 학습하고, 10, 30, 100으로 차례로 바꿔 다시 실행합니다
G_COUNT = 5
# 5, 10, 30, 100 중 선택. 추가로 학습할 G사 학습셋 건수입니다.
# 적은 건수부터 차례로 실행해 G사 정확도가 언제부터 충분히 오르는지 확인합니다. 건수가 많을수록 학습 시간이 늘어납니다.

update_and_check(G_COUNT, old_count=0)

curve = sorted([r for r in m3_records if r['기존 건수'] == 0], key=lambda r: r['G사 건수'])
if len(curve) >= 2:
    plt.figure(figsize=(6, 3.5))
    plt.plot([r['G사 건수'] for r in curve], [r['G사(%)'] for r in curve], marker='o', label='G')
    plt.plot([r['G사 건수'] for r in curve], [r['A~F사(%)'] for r in curve], marker='o', label='A-F')
    plt.xlabel('G samples added')   # 그래프 글자는 한글 글꼴이 없는 환경을 고려해 영어로 씁니다
    plt.ylabel('exact match (%)')
    plt.ylim(0, 100)
    plt.legend()
    plt.show()

**Q1.** 학습 곡선을 보고, G사 학습셋을 몇 건까지 모으면 충분하다고 판단되는지 적어 주세요.

**답:** (이 칸을 두 번 클릭해 적습니다)

## 3단계: 기존 학습셋을 섞어 망각 줄이기

2단계에서 G사만 학습했을 때 A\~F사 정확도가 학습 전보다 떨어졌다면, 새 유형을 학습하면서 기존에 학습한 것을 잃은 것입니다(망각). G사 학습셋에 기존 A\~F사 학습셋을 함께 섞어 학습하면 A\~F사 정확도가 유지되는지 확인해 봅시다.

In [ ]:
# ✏️ 직접 고칠 곳: G_COUNT에는 2단계에서 충분하다고 판단한 건수를 넣고, OLD_COUNT를 100과 300으로 바꿔 실행합니다
G_COUNT = 30
# 2단계에서 충분하다고 판단한 G사 건수를 넣습니다.

OLD_COUNT = 100
# 100, 300 중 선택. 함께 섞을 기존 A~F사 학습셋 건수입니다.
# 많이 섞을수록 기존 정확도를 지키기 쉽지만 학습 시간이 늘어납니다.

update_and_check(G_COUNT, OLD_COUNT)

**Q2.** 기록표에서 같은 G사 건수로 기존 학습셋을 섞지 않았을 때와 섞었을 때의 A\~F사 정확도를 비교하고, 차이가 난 이유를 적어 주세요.

**답:** (이 칸을 두 번 클릭해 적습니다)

## 확장 미션(선택): 내 업무에 새 유형이 추가된 경우로 해 보기

미션 1 확장 미션에서 파인튜닝한 내 업무 모델에 새 유형이 추가되는 상황을 정하고, 같은 과정을 해 봅니다.

1. 새 유형의 가상 재료를 Claude Code로 만들고, 시스템 프롬프트에 새 유형 규칙을 추가해 학습셋을 만듭니다.
2. 이 노트북의 `update_and_check()`처럼 새 유형 건수를 늘려 가며 학습 곡선을 확인합니다.
3. 기존 학습셋을 섞었을 때와 섞지 않았을 때의 기존 유형 정확도를 비교합니다.

**회고:** 비교 결과 등 자유롭게 적어 주세요. (이 칸을 두 번 클릭해 적습니다)

## 마치며

미션 1, 2, 3을 마친 노트북을 zip으로 묶어 휴넷 랩스에 올립니다.

1. Q 답 칸과 확장 미션 회고 칸이 채워졌는지 훑어봅니다.
2. [File] → [Save Notebook]으로 실행 결과와 답을 저장합니다.
3. 왼쪽 파일 창에서 `6일차_실습미션.ipynb`를 오른쪽 클릭 → [Download]로 내려받습니다.
4. 파일 이름을 `아이디_6일차_실습미션.ipynb`로 바꿉니다. 예: `aaa_6일차_실습미션.ipynb`
5. 휴넷 랩스에는 .ipynb 파일이 바로 올라가지 않으므로 zip으로 묶습니다. 내 PC에서 파일을 오른쪽 클릭해 압축 메뉴(Windows는 [ZIP 파일로 압축] 또는 [보내기] → [압축(ZIP) 폴더], Mac은 [압축])를 고르고, 만든 zip 파일을 [휴넷 랩스](https://dssw.labs.hunet.com/)에 올립니다.